## Sales Trends Monitoring:
### Goal: 
Generate time-based summaries to analyze sales patterns.
### Why it matters: 
Helps identify peak periods and plan resources.
### How to do it:
- Aggregate daily, weekly, and monthly revenue from order
- Break down by:
- • Location
- • Menu category (if available)
- • Time of day (optional)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
df_fact_order = spark.read.table('global_partner_project.gold.fact_order')
df_dim_date = spark.read.table('global_partner_project.gold.dim_date')

In [0]:
#print(df_fact_order.count())

In [0]:
df_order = df_fact_order.join(df_dim_date,
                              on='date_key',
                              how='inner')
#print(df_fact_order.count())

In [0]:
df_order.printSchema()

In [0]:
daily_agg_df = df_order.groupBy('date_key').agg(F.round(F.sum('item_price'),3).alias('total_revenue')).orderBy(F.to_date(F.col('date_key'),'dd-MM-yyyy'))
#daily_agg_df.display()

In [0]:
weekly_agg_df = df_order.groupBy('year','month','week').agg(F.round(F.sum('item_price'),3).alias('total_revenue')).orderBy('year','month','week')
#weekly_agg_df.display()

In [0]:
monthly_agg_df = df_order.groupBy('year','month').agg(F.round(F.sum('item_price'),3).alias('total_revenue')).orderBy('year','month')
#monthly_agg_df.display()

In [0]:
df_dim_item = spark.read.table('global_partner_project.gold.dim_item')

In [0]:
df_item_order = df_order.join(df_dim_item,
                              on='item_id',
                              how='inner')
#print(df_item_order.count())

In [0]:
agg_df_item_category = df_item_order.groupBy('item_category').agg(F.round(F.sum('item_price'),3).alias('total_revenue')).orderBy(F.col('total_revenue').desc())
#agg_df_item_category.display()

In [0]:
%sql
DROP TABLE IF EXISTS global_partner_project.mart.daily_revenue;
DROP TABLE IF EXISTS global_partner_project.mart.weekly_revenue;
DROP TABLE IF EXISTS global_partner_project.mart.monthly_revenue;
DROP TABLE IF EXISTS global_partner_project.mart.item_category_revenue;

In [0]:
daily_agg_df.write.mode('append').saveAsTable('global_partner_project.mart.daily_revenue')
weekly_agg_df.write.mode('append').saveAsTable('global_partner_project.mart.weekly_revenue')
monthly_agg_df.write.mode('append').saveAsTable('global_partner_project.mart.monthly_revenue')
agg_df_item_category.write.mode('append').saveAsTable('global_partner_project.mart.item_category_revenue')